In [1]:
from dotenv import load_dotenv
load_dotenv()


True

In [2]:
from openai import OpenAI
import os

In [3]:
client = OpenAI(
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
    api_key=os.environ.get("GEMINI_API_KEY")
)

In [4]:
from embedder import Embedder

embed = Embedder()

q1 = "Can I still join the course after the start date?"
q2 = "How to install Docker on Windows?"
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."

v1 = embed.encode(q1)
v2 = embed.encode(q2)
dv = embed.encode(d)

2026-06-27 16:16:51.995289943 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


In [5]:
v1.dot(dv)

np.float64(0.3233238799303238)

In [6]:
v2.dot(dv)

np.float64(0.019730422395141473)

In [7]:
qhw = "How does approximate nearest neighbor search work?"

In [8]:
hwv = embed.encode(qhw)
print(hwv.shape)    
print(hwv[0])

(384,)
-0.02058203437252893


In [9]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]

In [10]:
len(documents)

72

In [11]:
target_doc = next(
    doc for doc in documents
    if "07-sqlitesearch-vector.md" in doc["filename"]
)

In [15]:
text = target_doc["content"]
v3 = embed.encode(text)

In [16]:
v3.dot(hwv)

np.float64(0.36107026789538205)

In [17]:
from gitsource import chunk_documents
chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

In [18]:
print(type(chunks[0]))

<class 'dict'>


In [19]:
from tqdm.auto import tqdm

In [20]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(chunks), batch_size)):
    batch = chunks[i:i + batch_size]
    texts = [c["content"] for c in batch]
    batch_vectors = embed.encode_batch(texts)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/6 [00:00<?, ?it/s]

295

In [21]:
import numpy as np
X = np.array(vectors)

In [22]:
X.shape

(295, 384)

In [23]:
scores = X.dot(hwv)

In [24]:
best_idx = np.argmax(scores)
print(best_idx)

94


In [25]:
print(chunks[best_idx]["filename"])

02-vector-search/lessons/07-sqlitesearch-vector.md


In [26]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["filename"])
vindex.fit(X, chunks)

In [29]:
query = "What metric do we use to evaluate a search engine?"
query_vector = embed.encode(query)

In [30]:
results = vindex.search(query_vector, num_results=1)

In [31]:
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup